# Data Vortex — Phase 2: SQL Challenge 8
## High-Impact Post Leaderboard

### 1. Challenge Description
Identify and rank the highest-impact individual posts across the platform based on total engagement (`likes + shares + comments`).

Rules & Objectives:
- Total engagement defined using `COALESCE` to prevent missing values from invalidating totals.
- Original likes, shares, and comments remain untouched.
- Return top 20 ranked by `total_engagement DESC`, then `likes DESC`, `shares DESC`, `comments DESC`, `post_id ASC`.
- Provide platform summary for the top 20 cohort (treating NULL platform as 'Unknown').
- Validate data integrity and verify database immutability.

In [ ]:
import os
import sqlite3
import pandas as pd

# File Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(BASE_DIR, "data", "data_vortex.db")
SQL_PATH = os.path.join(BASE_DIR, "sql", "challenge_08_high_impact_post_leaderboard.sql")

print(f"Target Database: {DB_PATH}")
print(f"SQL Script:      {SQL_PATH}")

# Connect to SQLite
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print("Connected to SQLite database successfully.")

### 2. Top-20 High-Impact Posts Leaderboard
Retrieves and ranks the top 20 posts with deterministic tie-breaking.

In [ ]:
q_top20 = """
WITH post_engagement AS (
    SELECT 
        post_id,
        user_id,
        platform,
        timestamp,
        likes,
        shares,
        comments,
        (COALESCE(likes, 0) + COALESCE(shares, 0) + COALESCE(comments, 0)) AS total_engagement,
        ROW_NUMBER() OVER (
            ORDER BY 
                (COALESCE(likes, 0) + COALESCE(shares, 0) + COALESCE(comments, 0)) DESC,
                likes DESC,
                shares DESC,
                comments DESC,
                post_id ASC
        ) AS rank
    FROM posts
)
SELECT 
    rank,
    post_id,
    user_id,
    platform,
    timestamp,
    likes,
    shares,
    comments,
    total_engagement
FROM post_engagement
WHERE rank <= 20
ORDER BY rank ASC;
"""

df_top20 = pd.read_sql_query(q_top20, conn)
df_top20

### 3. Platform Distribution Summary for Top 20 Cohort
Aggregates top-20 post counts and average total engagement by platform (NULL platforms grouped as 'Unknown').

In [ ]:
q_platform = """
WITH post_engagement AS (
    SELECT 
        post_id,
        platform,
        (COALESCE(likes, 0) + COALESCE(shares, 0) + COALESCE(comments, 0)) AS total_engagement,
        ROW_NUMBER() OVER (
            ORDER BY 
                (COALESCE(likes, 0) + COALESCE(shares, 0) + COALESCE(comments, 0)) DESC,
                likes DESC,
                shares DESC,
                comments DESC,
                post_id ASC
        ) AS rank
    FROM posts
),
top20 AS (
    SELECT * 
    FROM post_engagement 
    WHERE rank <= 20
)
SELECT 
    COALESCE(platform, 'Unknown') AS platform,
    COUNT(*) AS number_of_top_posts,
    ROUND(AVG(total_engagement), 2) AS average_total_engagement
FROM top20
GROUP BY COALESCE(platform, 'Unknown')
ORDER BY number_of_top_posts DESC, average_total_engagement DESC;
"""

df_platform = pd.read_sql_query(q_platform, conn)
df_platform

### 4. Validation Checks
Verifies data integrity across required challenge benchmarks.

In [ ]:
# Validation 1: Database contains exactly 12,000 posts
db_posts = conn.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
print(f"1. Total posts in DB:         {db_posts} (Expected: 12000) -> {'PASS' if db_posts == 12000 else 'FAIL'}")

# Validation 2: Top leaderboard contains exactly 20 posts
num_top = len(df_top20)
print(f"2. Top leaderboard row count: {num_top} (Expected: 20) -> {'PASS' if num_top == 20 else 'FAIL'}")

# Validation 3: Post IDs in top 20 are unique
unique_pids = df_top20['post_id'].nunique()
print(f"3. Unique post IDs in top 20: {unique_pids} (Expected: 20) -> {'PASS' if unique_pids == 20 else 'FAIL'}")

# Validation 4: Total engagement calculation accuracy
calc_correct = all(
    df_top20['total_engagement'] == (
        df_top20['likes'].fillna(0) + df_top20['shares'].fillna(0) + df_top20['comments'].fillna(0)
    )
)
print(f"4. Total engagement math:     {calc_correct} (Expected: True) -> {'PASS' if calc_correct else 'FAIL'}")

# Validation 5: Cleaned CSV remains identical
df_clean = pd.read_csv(os.path.join(BASE_DIR, "data", "cleaned", "Social_Engine_Posts_Cleaned.csv"))
print(f"5. Cleaned CSV row count:     {len(df_clean)} (Expected: 12000) -> {'PASS' if len(df_clean) == 12000 else 'FAIL'}")

In [ ]:
# Close connection
conn.close()
print("Database connection closed cleanly.")